# 1 - Explore the datasets

We start from three publicly available datasets, in the formats in which they are distributed:

| Dataset                      | Version/Year | Resolution |  Provider                   | Original format             |
|------------------------------|--------------|------------|-----------------------------|-----------------------------|
| AHN Digital Surface Model    | 4            | 0.5 m      | [PDOK / AHN][AHN]           | GeoTIFF tiles (1 km x 1 km) |
| Aerial images NL (RGB)       | 2024         | 8 cm       | [PDOK / Beeldmateriaal][BM] | GeoTIFF tiles (1 km x 1 km) |
| Daily mean temperature (Tg1) | 5            | 1 km       | [KNMI][Tg1]                 | NetCDF3, one file per day   |

[AHN]: https://www.ahn.nl/
[BM]: https://www.beeldmateriaal.nl/
[Tg1]: https://dataplatform.knmi.nl/dataset/tg1-5 

A copy of a small subset of them is on the object store. Which properties make these files more or less suitable for cloud-native access?

In [1]:
import os

import fsspec

from utils import ENDPOINT_URL, BUCKET_URI, print_raster_info, print_netcdf_info

In [2]:
fs = fsspec.filesystem("s3", anon=True, endpoint_url=ENDPOINT_URL)   

In [3]:
print(fs.tree(BUCKET_URI, recursion_limit=1))

s3://cng-forum-workshop
├── AHN-4
├── KNMI-Tg1-5
└── aerial-images-NL-2024


## AHN Digital Surface Model

In [4]:
print(fs.tree(f"{BUCKET_URI}/AHN-4", recursion_limit=2, max_display=5))

s3://cng-forum-workshop/AHN-4
└── DSM05
    ├── AHN4_R_135000_454000.TIF
    ├── AHN4_R_135000_455000.TIF
    ├── AHN4_R_135000_456000.TIF
    ├── AHN4_R_135000_457000.TIF
    ├── AHN4_R_136000_454000.TIF
    └── 11 more item(s) not displayed.


In [5]:
dsm_path, *_ = fs.glob(f"{BUCKET_URI}/AHN-4/DSM05/*.TIF")
print_raster_info(dsm_path)

URL:          s3://cng-forum-workshop/AHN-4/DSM05/AHN4_R_135000_454000.TIF
Driver:       GTiff
Size:         2000 x 2000 , Bands: 1
CRS:          EPSG:7415
Bounds:       BoundingBox(left=135000.0, bottom=454000.0, right=136000.0, top=455000.0)
Resolution:   (0.5, 0.5)
Dtypes:       ('float32',)
Nodata:       -9999.0
Block shapes: [(256, 256)]
Tiled:        True
Compression:  DEFLATE
Layout:       None
Bands:       
 * Band 1: description=Hoogte (m NAP), overviews=[]


The GeoTIFF is already internally tiled (256 x 256 blocks) and compressed (DEFLATE), but it has **no overviews** and GDAL doesn't recognise it as a COG (`Layout: None`).

## RGB Aerial Images

In [6]:
print(fs.tree(f"{BUCKET_URI}/aerial-images-NL-2024", recursion_limit=2, max_display=5))

s3://cng-forum-workshop/aerial-images-NL-2024
└── RGB
    ├── 2024_135000_455000_RGB_hrl.tif
    ├── 2024_135000_456000_RGB_hrl.tif
    ├── 2024_135000_457000_RGB_hrl.tif
    ├── 2024_135000_458000_RGB_hrl.tif
    ├── 2024_136000_455000_RGB_hrl.tif
    └── 11 more item(s) not displayed.


In [7]:
rgb_path, *_ = fs.glob(f"{BUCKET_URI}/aerial-images-NL-2024/RGB/*.tif")
print_raster_info(rgb_path)

URL:          s3://cng-forum-workshop/aerial-images-NL-2024/RGB/2024_135000_455000_RGB_hrl.tif
Driver:       GTiff
Size:         12500 x 12500 , Bands: 3
CRS:          EPSG:28992
Bounds:       BoundingBox(left=135000.0, bottom=454000.0, right=136000.0, top=455000.0)
Resolution:   (0.08, 0.08)
Dtypes:       ('uint8', 'uint8', 'uint8')
Nodata:       None
Block shapes: [(256, 256), (256, 256), (256, 256)]
Tiled:        True
Compression:  JPEG
Layout:       None
Bands:       
 * Band 1: description=None, overviews=[]
 * Band 2: description=None, overviews=[]
 * Band 3: description=None, overviews=[]


Tiled and JPEG-compressed, but 12500 x 12500 pixels without overviews: an overview-level image of a tile requires reading the entire file!

## Gridded Daily Mean Temperature

In [8]:
print(fs.tree(f"{BUCKET_URI}/KNMI-Tg1-5", recursion_limit=2, max_display=5))

s3://cng-forum-workshop/KNMI-Tg1-5
├── 20250101
│   └── INTER_OPER_R___TG1_____L3__20250101T000000_20250102T000000_0006.nc
├── 20250102
│   └── INTER_OPER_R___TG1_____L3__20250102T000000_20250103T000000_0006.nc
├── 20250103
│   └── INTER_OPER_R___TG1_____L3__20250103T000000_20250104T000000_0006.nc
├── 20250104
│   └── INTER_OPER_R___TG1_____L3__20250104T000000_20250105T000000_0006.nc
├── 20250105
│   └── INTER_OPER_R___TG1_____L3__20250105T000000_20250106T000000_0006.nc
└── 360 more item(s) not displayed.


In [9]:
nc_path, *_ = fs.glob(f"{BUCKET_URI}/KNMI-Tg1-5/*/*.nc")
print_netcdf_info(nc_path)

URL:          https://objectstore.surf.nl/cng-forum-workshop/KNMI-Tg1-5/20250101/INTER_OPER_R___TG1_____L3__20250101T000000_20250102T000000_0006.nc
Format:       NETCDF3_CLASSIC
Dimensions:   {'x': 300, 'y': 350, 'time': 1}
Variables:   
 * x: dtype=float64, dims=('x',), shape=(300,), chunking=None, filters=None
 * y: dtype=float64, dims=('y',), shape=(350,), chunking=None, filters=None
 * lon: dtype=float32, dims=('y', 'x'), shape=(350, 300), chunking=None, filters=None
 * lat: dtype=float32, dims=('y', 'x'), shape=(350, 300), chunking=None, filters=None
 * time: dtype=float64, dims=('time',), shape=(1,), chunking=None, filters=None
 * prediction: dtype=float32, dims=('time', 'y', 'x'), shape=(1, 350, 300), chunking=None, filters=None
 * stationvalues: dtype=float32, dims=('time', 'y', 'x'), shape=(1, 350, 300), chunking=None, filters=None
 * stations: dtype=float32, dims=('time', 'y', 'x'), shape=(1, 350, 300), chunking=None, filters=None
 * iso_dataset: dtype=|S1, dims=(), shape=(),

NetCDF3: no chunking, no compression. One small file per day: fine to get a map for a given day, but extracting the time series at one location means touching 365 files.

> ## Discussion
> For each dataset, which access pattern you think users will need most? What you would change in the file layout to support it?